In [40]:
import torch
from torchvision import datasets, transforms


# 1. Pehle images ko preprocess karte hain (Resize, Tensor, Normalize)
# CNN ko hamesha same size ki images chahiye hoti hain, isliye Resize(128, 128) kar rahe hain
transform = transforms.Compose([
    transforms.Resize((128, 128)), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


dataset=datasets.ImageFolder(root="./PetImages", transform=transform)

print("Classes:", dataset.classes)       # Output: ['Cat', 'Dog']
print("Class to Index:", dataset.class_to_idx) # Output: {'Cat': 0, 'Dog': 1}
print("Total Images:", len(dataset))

train_loader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True)
val_loader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True)

Classes: ['Cat', 'Dog']
Class to Index: {'Cat': 0, 'Dog': 1}
Total Images: 24998


## Import Librarys

In [41]:
import torch
import torch.nn as nn 
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision import datasets, models
import os
import time
import copy


In [42]:
## CNN Model Defination
class CATDOGCLASSIFY(nn.Module):
    def __init__(self):
        super(CATDOGCLASSIFY, self).__init__()

        # Convolutional Layers
        self.conv1 = nn.Conv2d(3, 32,kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128 , kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=2, padding=1)


        # Pooling
        self.pool = nn.MaxPool2d(2, 2)
        
        # Dropout for regularization
        self.dropout = nn.Dropout(0.5)

        # Fully connected Layers

        self.fc1 = nn.Linear(256 * 16 * 16, 512)  # Assuming 256x256 input -> 16x16 after pooling
        self.fc2 = nn.Linear(512, 128)
        self.fc3 = nn.Linear(128, 2)  # 2 classes: Cat and Dog
        
        # Activation
        self.relu = nn.ReLU()
    
    def forward(self, x):
        # Convolutional blocks
        x = self.pool(self.relu(self.conv1(x)))  # 256x256 -> 128x128
        x = self.pool(self.relu(self.conv2(x)))  # 128x128 -> 64x64
        x = self.pool(self.relu(self.conv3(x)))  # 64x64 -> 32x32
        x = self.pool(self.relu(self.conv4(x)))  # 32x32 -> 16x16
        
        # Flatten
        x = torch.flatten(x, 1) 
         # Update fc1 if needed (only once)
        if not hasattr(self, '_fc1_updated'):
            expected_features = x.shape[1]
            if expected_features != self.fc1.in_features:
                self.fc1 = nn.Linear(expected_features, 512)
                self.fc1 = self.fc1.to(x.device)
                self._fc1_updated = True
        # Fully connected layers with dropout
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.dropout(self.relu(self.fc2(x)))
        x = self.fc3(x)
        
        return x
    

In [43]:
# 2. Training Function
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=25, device='cuda'):
    since = time.time()
    
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    
    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)
        
        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
                dataloader = train_loader
            else:
                model.eval()   # Set model to evaluate mode
                dataloader = val_loader
            
            running_loss = 0.0
            running_corrects = 0
            
            # Iterate over data
            for inputs, labels in dataloader:
                inputs = inputs.to(device)
                labels = labels.to(device)
                
                # Zero the parameter gradients
                optimizer.zero_grad()
                
                # Forward
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    
                    # Backward + optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                
                # Statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
            
            if phase == 'train':
                scheduler.step()
            
            epoch_loss = running_loss / len(dataloader.dataset)
            epoch_acc = running_corrects.double() / len(dataloader.dataset)
            
            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
            
            # Deep copy the model
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
        
        print()
    
    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:4f}')
    
    # Load best model weights
    model.load_state_dict(best_model_wts)
    return model

In [44]:
# 3. Main Execution
def main():
    # Set device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device: {device}')
    
    # Data transforms
    data_transforms = {
        'train': transforms.Compose([
            transforms.RandomResizedCrop(256),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ]),
        'val': transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(256),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ]),
    }
    
    # Create datasets (assuming you have train_loader and val_loader already)
    # If not, you can create them like this:
    # image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x]) for x in ['train', 'val']}
    # dataloaders = {x: DataLoader(image_datasets[x], batch_size=32, shuffle=True, num_workers=4) for x in ['train', 'val']}
    
    # Initialize model
    model = CATDOGCLASSIFY().to(device)
    
    # Loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
    
    # Learning rate scheduler
    exp_lr_scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)
    
    # Train the model
    model = train_model(
        model, 
        train_loader,  # Your existing train_loader
        val_loader,    # Your existing val_loader
        criterion, 
        optimizer, 
        exp_lr_scheduler,
        num_epochs=25,
        device=device
    )
    
    # Save the model
    torch.save(model.state_dict(), 'cat_dog_classifier.pth')
    print('Model saved as cat_dog_classifier.pth')
    
    return model


In [45]:
def predict_image(model, image_path, device='cuda'):
    model.eval()
    
    # Load and preprocess image
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(256),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    from PIL import Image
    img = Image.open(image_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0).to(device)
    
    # Make prediction
    with torch.no_grad():
        outputs = model(img_tensor)
        _, predicted = torch.max(outputs, 1)
        probs = torch.nn.functional.softmax(outputs, dim=1)
    
    classes = ['Cat', 'Dog']
    predicted_class = classes[predicted.item()]
    confidence = probs[0][predicted.item()].item()
    
    return predicted_class, confidence

In [ ]:
if __name__ == '__main__':
    trained_model = main()

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from PIL import ImageFile
from tqdm import tqdm  # Progress bar ke liye
import time
import copy
import os

# 🛑 FIX 1: Corrupt Images ko handle karna
# Kaggle dataset mein kuch images adhuri (truncated) hoti hain. 
# Yeh line PIL ko bolti hai ke "Jo image mili hai usko load kar lo, crash mat karo"
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ==========================================
# 1. MODEL ARCHITECTURE (Size-Agnostic)
# ==========================================
class CatDogCNN(nn.Module):
    def __init__(self):
        super(CatDogCNN, self).__init__()
        
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        
        # 🛑 FIX 2: Adaptive Pooling (Design Pattern from ML Books)
        # Isse farq nahi padta image ka size kya hai, yeh hamesha 4x4 bana dega
        # Isse aapka purana "batch_size mismatch" wala error kabhi nahi aayega!
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),  # 256 channels * 4 * 4
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.adaptive_pool(x)
        x = self.classifier(x)
        return x

# ==========================================
# 2. TRAINING LOOP WITH PROGRESS BAR
# ==========================================
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=5, device='cpu'):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'\n🚀 Epoch {epoch+1}/{num_epochs}')
        print('-' * 30)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                dataloader = train_loader
            else:
                model.eval()
                dataloader = val_loader

            running_loss = 0.0
            running_corrects = 0

            # FIX 3: tqdm se Progress Bar lagaya taake pata chale code hang nahi hai
            pbar = tqdm(dataloader, desc=f"{phase.capitalize()}ing", leave=False)
            
            for inputs, labels in pbar:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                
                # Progress bar par live loss dikhao
                pbar.set_postfix({'loss': f'{loss.item():.3f}'})

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / len(dataloader.dataset)
            epoch_acc = running_corrects.double() / len(dataloader.dataset)

            print(f'{phase} Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
                print(f'🏆 New Best Model Saved! (Acc: {best_acc:.4f})')

    print(f'\n Training Complete! Best Val Acc: {best_acc:4f}')
    model.load_state_dict(best_model_wts)
    return model

# ==========================================
# 3. MAIN EXECUTION
# ==========================================
def main():
    # Device setup
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f' Using device: {device}')

    # Data transforms (128x128 rakha hai taake CPU par bhi 5-10 min mein train ho jaye)
    data_transforms = {
        'train': transforms.Compose([
            transforms.RandomResizedCrop(128),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ]),
        'val': transforms.Compose([
            transforms.Resize(128),
            transforms.CenterCrop(128),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ]),
    }

    # Dataset Load (Assuming 'PetImages' folder has 'Cat' and 'Dog' subfolders)
    dataset_path = './PetImages'  # Apna path yahan dalein
    print(" Loading Dataset...")
    full_dataset = datasets.ImageFolder(dataset_path, transform=data_transforms['train'])
    
    # 80-20 Split
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
    
    # Validation dataset par val_transform apply karne ka jugad
    val_dataset.dataset = datasets.ImageFolder(dataset_path, transform=data_transforms['val'])

    # FIX 4: Windows/Jupyter Hang Issue
    # num_workers=0 rakhna zaroori hai Windows/Jupyter mein, warna code silently hang ho jata hai
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

    print(f" Train Images: {train_size} | Val Images: {val_size}")

    # Model, Loss, Optimizer
    model = CatDogCNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001) # Adam is better than SGD here
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

    # Train!
    model = train_model(
        model, train_loader, val_loader, 
        criterion, optimizer, scheduler, 
        num_epochs=10, device=device
    )

    # Save
    torch.save(model.state_dict(), 'cat_dog_best.pth')
    print('💾 Model saved as cat_dog_best.pth')

if __name__ == '__main__':
    main()